# 19 · FHIR Bundle Export + US Core Validation (v0.3.0)

End-to-end walk through the v0.3.0 FHIR interoperability path:

1. Build a small set of FHIR resources.
2. Validate them against **US Core 6.1.0**.
3. Export as a transaction **Bundle** (single JSON).
4. Export as **NDJSON** (FHIR Bulk Data `$export` shape).
5. Round-trip both shapes through `fhir.resources` to confirm conformance.

Install the FHIR extra first if you haven't:

```bash
pip install "portiere-health[fhir]"
```

Full reference: [`docs/fhir-profile-validation.md`](../fhir-profile-validation.md) and [`docs/fhir-bundle-export.md`](../fhir-bundle-export.md).

In [ ]:
import json
from pathlib import Path

import portiere

print('portiere version:', portiere.__version__)

## 1 · Build sample FHIR resources

Two compliant resources: a Patient and a referenced Observation. These are the same shapes Portiere produces from `Project.run_etl(..., target_model='fhir_r4')`.

In [ ]:
resources = [
    {
        'resourceType': 'Patient',
        'id': 'p1',
        'identifier': [
            {'system': 'urn:oid:2.16.840.1.113883.4.1', 'value': '111-22-3333'}
        ],
        'name': [{'family': 'Doe', 'given': ['Jane']}],
        'gender': 'female',
        'birthDate': '1990-01-15',
    },
    {
        'resourceType': 'Observation',
        'id': 'o1',
        'status': 'final',
        'code': {'text': 'Body height'},
        'subject': {'reference': 'Patient/p1'},
    },
]

print(f'{len(resources)} resources')

## 2 · Validate against US Core 6.1.0

v0.3.0 validates 10 resource types. Observation is included; the resource we built above lacks some Observation-specific required fields (we'll see them in the failure list). Patient should pass.

In [ ]:
from portiere.quality.fhir_profile.us_core import validate_against_us_core

report = validate_against_us_core(resources)

print(f'profile          : {report.profile}')
print(f'total resources  : {report.total_resources}')
print(f'passed           : {report.passed}')
print(f'failures         : {len(report.failures)}')
print(f'skipped          : {report.skipped}')
print()
for f in report.failures:
    print(f'  [{f.severity}] {f.resource_type}#{f.resource_index} '
          f'{f.invariant_id}: {f.message}')

The Observation lacks required US Core fields (e.g. `category`, a coded `code`). For the export demo below, that's fine — `portiere export` writes resources as-is unless `--fhir-profile` is passed.

If you *want* the export to refuse non-conformant input, pass `--fhir-profile us-core-6.1.0` to the CLI or pre-check with `project.validate(fhir_profile=...)` in Python.

## 3 · Export as a transaction Bundle

Each entry gets a fresh `urn:uuid:` `fullUrl` and a `POST` request.

In [ ]:
from portiere.export.fhir.bundle import to_transaction_bundle

bundle = to_transaction_bundle(resources)

print(f"resourceType : {bundle['resourceType']}")
print(f"type         : {bundle['type']}")
print(f"entries      : {len(bundle['entry'])}")
print()
print('First entry:')
print(json.dumps(bundle['entry'][0], indent=2))

## 4 · Export as NDJSON

One file per `resourceType`, one resource per line. This matches the FHIR Bulk Data `$export` operation output, so downstream `$import` tooling consumes it directly.

In [ ]:
import tempfile
from portiere.export.fhir.ndjson import to_ndjson_files

with tempfile.TemporaryDirectory() as tmp:
    out_dir = Path(tmp) / 'ndjson_out'
    files = to_ndjson_files(resources, out_dir=out_dir)
    print('Files written:')
    for p in files:
        print(f'  {p.name}  ({p.stat().st_size} bytes)')
    print()
    print('First line of Patient.ndjson:')
    print((out_dir / 'Patient.ndjson').read_text().splitlines()[0])

## 5 · Round-trip through `fhir.resources`

Both the Bundle and the NDJSON lines parse back through Pydantic v2 models — this is the same check `tests/test_fhir_export.py::TestFhirExportRoundTrip` runs in CI.

In [ ]:
from fhir.resources.bundle import Bundle
from fhir.resources import get_fhir_model_class

bundle_model = Bundle.model_validate(bundle)
print(f'Bundle.type           : {bundle_model.type}')
print(f'len(Bundle.entry)     : {len(bundle_model.entry)}')
print(f'entry[0].request.method : {bundle_model.entry[0].request.method}')

# Round-trip each resource directly through its model:
for r in resources:
    cls = get_fhir_model_class(r['resourceType'])
    parsed = cls.model_validate(r)
    print(f'  {r["resourceType"]:<15} -> {type(parsed).__name__} (id={parsed.id})')

## 6 · Plug into a FHIR server

The HAPI FHIR test server accepts transaction bundles directly:

```bash
curl -X POST \
  -H 'Content-Type: application/fhir+json' \
  --data-binary @bundle.json \
  https://hapi.fhir.org/baseR4
```

For NDJSON, point a FHIR `$import` operation at the directory.

The full CLI surface:

```bash
portiere export --input resources.json --format bundle  --out bundle.json
portiere export --input resources.json --format ndjson  --out ndjson_dir/
portiere export --input resources.json --format bundle  --out bundle.json \
    --fhir-profile us-core-6.1.0   # validate before writing
```